In [4]:
# Popalgo — Cell 1 (Core setup)

# This cell sets up the environment for the entire notebook.
# It defines project paths, API settings, season parameters,
# and a few helper utilities used throughout the rest of the pipeline.
#
# It also includes some safeguards to prevent data leakage when
# running backtests (making sure future games are never used
# when building historical features).

from __future__ import annotations

import os
import time
import unicodedata
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional, Dict, List

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Load the .env file from the project root
ENV_PATH = Path(r"C:\Users\djbar\popalgo\.env")
load_dotenv(dotenv_path=ENV_PATH)

ODDS_API_KEY = os.getenv("ODDS_API_KEY")
assert ODDS_API_KEY, f"ODDS_API_KEY not found. Checked: {ENV_PATH}"

ODDS_BASE_URL = "https://api.the-odds-api.com/v4"
ODDS_SPORT_KEY = "basketball_ncaab"
ODDS_REGIONS = "us"
ODDS_MARKETS = "totals,spreads"
ODDS_ODDS_FORMAT = "american"
ODDS_DATE_FORMAT = "iso"


# Global settings for the backtest and model configuration.
# The key assumption here is that any slate built for a given date
# can only use information available prior to that date.
LEAKAGE_STRICT: bool = True

DEFAULT_LOOKBACK_DAYS: int = 200
LOOKBACK_DAYS: int = DEFAULT_LOOKBACK_DAYS

SEASON_YEAR: int = 2026
FEATURE_VERSION: str = "v1"

MIN_GAMES: int = 3
BACKTEST_START_DATE: str = "2025-11-01"
BACKTEST_END_DATE: str = "2026-03-05"

# Some later cells reference this name for evaluation windows
EVAL_START_DATE: str = BACKTEST_START_DATE

# Using Eastern time helps avoid issues when ESPN returns
# timestamps close to midnight UTC.
SLATE_TZ = "America/New_York"


# HTTP settings used when requesting data from APIs.
# Retries and backoff help avoid occasional API hiccups.
SLEEP_S: float = 0.35
HTTP_TIMEOUT: int = 30
MAX_RETRIES: int = 3
RETRY_BACKOFF: float = 1.8

SCOREBOARD_URL: str = (
    "https://site.api.espn.com/apis/site/v2/sports/"
    "basketball/mens-college-basketball/scoreboard"
)

TEAMS_URL: str = (
    "https://site.api.espn.com/apis/site/v2/sports/"
    "basketball/mens-college-basketball/teams"
)


# Define the main project folder and all cache directories.
# Keeping everything under one root directory makes the project
# easier to move or run on another machine.
PROJECT_ROOT = Path(r"C:\Users\djbar\popalgo")

# Existing caches used by other parts of the project
EVA_CACHE_DIR = PROJECT_ROOT / "eva_cache"
TEAM_CACHE_DIR = PROJECT_ROOT / "team_cache"
MATCHUP_CACHE_DIR = PROJECT_ROOT / "matchup_cache"

# Season-specific output folders
CACHE_CBB_DIR = PROJECT_ROOT / "cache_cbb" / f"season_{SEASON_YEAR}"
ODDS_CACHE_DIR = PROJECT_ROOT / "odds_cache" / f"season_{SEASON_YEAR}" / ODDS_SPORT_KEY

# Mapping between ESPN team names and Odds API team names
OFFICIAL_CROSSWALK_PATH = PROJECT_ROOT / "official_odds_espn_team_crosswalk.csv"

# Ensure cache directories exist so later cells can write files safely
for p in (EVA_CACHE_DIR, TEAM_CACHE_DIR, MATCHUP_CACHE_DIR, CACHE_CBB_DIR, ODDS_CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)


# Helper functions for consistent date formatting

def iso_date(x: Any) -> str:
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def yyyymmdd(x: Any) -> str:
    return pd.to_datetime(x).strftime("%Y%m%d")


def normalize_date(x: Any) -> pd.Timestamp:
    return pd.to_datetime(x).normalize()


def date_range_str(start_date: str, end_date: str) -> List[str]:
    rng = pd.date_range(normalize_date(start_date), normalize_date(end_date), freq="D")
    return [d.strftime("%Y-%m-%d") for d in rng]


# Small logging helper used across the notebook

def log_line(msg: str) -> None:
    ts = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")


def compact_exception(e: Exception, max_chars: int = 500) -> str:
    s = repr(e)
    return s if len(s) <= max_chars else s[:max_chars] + "..."


# Backtest safety checks to ensure no future information leaks
# into historical feature calculations.

def assert_feature_cutoff(game_date: Any, as_of_date: Any) -> None:
    """
    Feature rows must come from games on or before the as-of cutoff date.
    Example: if a slate is built for 1/12, the feature window should end
    on 1/11 or earlier.
    """
    if not LEAKAGE_STRICT:
        return

    gd = normalize_date(game_date)
    ao = normalize_date(as_of_date)

    assert gd <= ao, f"Leakage detected: game_date {gd.date()} must be <= as_of_date {ao.date()}"


def assert_slate_alignment(slate_date: Any, as_of_date: Any) -> None:
    """
    Ensures the as-of date always comes before the slate date.
    """
    if not LEAKAGE_STRICT:
        return

    sd = normalize_date(slate_date)
    ao = normalize_date(as_of_date)

    assert ao < sd, f"Leakage detected: as_of_date {ao.date()} must be < slate_date {sd.date()}"


@dataclass(frozen=True)
class RunContext:
    """
    Simple container that keeps track of configuration
    for a single slate run.
    """

    slate_date: str
    as_of_date: str
    season_year: int = SEASON_YEAR
    lookback_days: int = LOOKBACK_DAYS
    feature_version: str = FEATURE_VERSION

    def __post_init__(self):
        if LEAKAGE_STRICT:
            assert_slate_alignment(self.slate_date, self.as_of_date)


# Helper for making API requests with retry logic

_session = requests.Session()


def http_get_json(
    url: str,
    params: Optional[dict] = None,
    timeout: int = HTTP_TIMEOUT,
    sleep_s: float = SLEEP_S,
) -> dict:

    last_err: Optional[Exception] = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            r = _session.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()

        except Exception as e:
            last_err = e
            log_line(f"HTTP error {attempt}/{MAX_RETRIES}: {e}")

            if attempt < MAX_RETRIES:
                time.sleep((RETRY_BACKOFF ** (attempt - 1)) * sleep_s)
            else:
                raise last_err


# Quick initialization message to confirm settings when the notebook runs

log_line(
    "CELL 1 loaded — "
    f"PROJECT_ROOT={PROJECT_ROOT} | "
    f"SEASON_YEAR={SEASON_YEAR} | "
    f"LOOKBACK_DAYS={LOOKBACK_DAYS} | "
    f"LEAKAGE_STRICT={LEAKAGE_STRICT} | "
    f"SLATE_TZ={SLATE_TZ}"
)

log_line(
    f"CROSSWALK exists? {OFFICIAL_CROSSWALK_PATH.exists()} | "
    f"file={OFFICIAL_CROSSWALK_PATH.name}"
)

[2026-03-08 18:34:15] CELL 1 loaded — PROJECT_ROOT=C:\Users\djbar\popalgo | SEASON_YEAR=2026 | LOOKBACK_DAYS=200 | LEAKAGE_STRICT=True | SLATE_TZ=America/New_York
[2026-03-08 18:34:15] CROSSWALK exists? True | file=official_odds_espn_team_crosswalk.csv


In [5]:
# Popalgo — Cell 2 (Master ingest)

# This cell builds and maintains the season-long master table of completed games.
# Each ESPN event is converted into two rows, one for each team.
#
# Only completed games with valid scores are kept here.
# This table acts as the season-level base dataset that later cells
# use for possessions, features, projections, and backtesting.

from __future__ import annotations

import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# Store the season-level master table in a dedicated folder.
# Keeping this separated by season makes reruns cleaner and safer.
MASTER_DIR = PROJECT_ROOT / "cache_cbb" / "master" / f"season_{SEASON_YEAR}"
MASTER_DIR.mkdir(parents=True, exist_ok=True)
MASTER_PATH = MASTER_DIR / "master_team_games.parquet"


# These are the fields that define the master table structure.
# Standardizing this up front helps keep the file consistent across reruns.
REQUIRED_COLS = [
    "season_year",
    "game_date",         # YYYY-MM-DD
    "event_id",          # ESPN event id
    "commence_time",     # ESPN competition timestamp
    "team_id",
    "team_name",
    "opponent_id",
    "opponent_name",
    "home_away",         # home / away / neutral
    "neutral_site",      # bool
    "points_for",
    "points_against",
    "game_completed",    # bool
    "status_desc",       # Final, Final/OT, etc.
]


def standardize_master(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and standardize the master table so the schema stays stable
    even if data is appended across multiple runs.
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=REQUIRED_COLS)

    out = df.copy()

    # Make sure every expected column exists
    for c in REQUIRED_COLS:
        if c not in out.columns:
            out[c] = np.nan

    # Keep ids as strings so joins and deduping behave consistently
    for c in ["event_id", "team_id", "opponent_id"]:
        out[c] = out[c].astype(str)
        out.loc[out[c].isin(["nan", "None", ""]), c] = ""

    # Normalize game dates to YYYY-MM-DD
    out["game_date"] = pd.to_datetime(out["game_date"], errors="coerce").dt.strftime("%Y-%m-%d")

    # Normalize booleans
    out["game_completed"] = out["game_completed"].fillna(False).astype(bool)
    out["neutral_site"] = out["neutral_site"].fillna(False).astype(bool)

    # Normalize scores
    out["points_for"] = pd.to_numeric(out["points_for"], errors="coerce")
    out["points_against"] = pd.to_numeric(out["points_against"], errors="coerce")

    # Remove rows missing core identifiers
    out = out[(out["event_id"] != "") & (out["team_id"] != "")].copy()

    # Final cleanup
    out.drop_duplicates(subset=["season_year", "event_id", "team_id"], inplace=True)
    out.sort_values(["game_date", "event_id", "team_id"], inplace=True)
    out.reset_index(drop=True, inplace=True)

    return out


def load_master() -> pd.DataFrame:
    """
    Load the season master table if it already exists.
    """
    if MASTER_PATH.exists():
        df = pd.read_parquet(MASTER_PATH)
        return standardize_master(df)

    return pd.DataFrame(columns=REQUIRED_COLS)


def save_master(df: pd.DataFrame) -> None:
    """
    Save the standardized master table back to disk.
    """
    MASTER_DIR.mkdir(parents=True, exist_ok=True)
    standardize_master(df).to_parquet(MASTER_PATH, index=False)


def fetch_scoreboard_all(game_date: str, page_limit: int = 8) -> dict:
    """
    Pull all scoreboard events for a given date from ESPN.

    ESPN can paginate large slates, so this keeps requesting pages,
    deduplicates by event id, and uses a hard page cap as a safety stop.
    """
    date_str = yyyymmdd(game_date)

    all_events: List[dict] = []
    seen_ids: set[str] = set()
    page = 1
    prev_seen_ct = 0

    while True:
        data = http_get_json(
            SCOREBOARD_URL,
            params={"dates": date_str, "groups": 50, "limit": 500, "page": page},
        )

        events = data.get("events", []) or []
        if not events:
            break

        for ev in events:
            eid = str(ev.get("id") or "")
            if eid and eid not in seen_ids:
                all_events.append(ev)
                seen_ids.add(eid)

        # If the page comes back short, we are likely done
        if len(events) < 500:
            break

        # Stop if ESPN keeps returning pages without any new events
        if len(seen_ids) == prev_seen_ct:
            log_line(f"Pagination stalled at page={page}; stopping.")
            break
        prev_seen_ct = len(seen_ids)

        # Safety cap in case pagination behaves strangely
        if page >= page_limit:
            log_line(f"Stopping pagination early at page={page} to avoid a hang.")
            break

        page += 1

    data["events"] = all_events
    return data


def _competitors(event: dict) -> List[dict]:
    comps = event.get("competitions") or []
    return comps[0].get("competitors", []) if comps else []


def event_to_team_rows(event: dict, game_date: str) -> List[Dict[str, Any]]:
    """
    Convert a single ESPN event into two team-level rows.

    Only completed games with usable scores are returned.
    """
    eid = str(event.get("id") or "")
    if not eid:
        return []

    comps = _competitors(event)
    if len(comps) != 2:
        return []

    comp0 = (event.get("competitions") or [{}])[0]
    commence_time = comp0.get("date")
    status = (comp0.get("status", {}).get("type", {}) or {})
    completed = bool(status.get("completed", False))
    status_desc = str(status.get("description") or "")

    neutral_site = bool(comp0.get("neutralSite", False))

    def parse_side(c: dict) -> Dict[str, Any]:
        team = c.get("team") or {}
        pts = pd.to_numeric(c.get("score"), errors="coerce")

        ha = (c.get("homeAway") or "").lower().strip()
        if neutral_site:
            ha = "neutral"
        if ha not in {"home", "away", "neutral"}:
            ha = ""

        return {
            "team_id": str(team.get("id") or ""),
            "team_name": str(team.get("displayName") or team.get("name") or ""),
            "home_away": ha,
            "points_for": pts,
        }

    a, b = parse_side(comps[0]), parse_side(comps[1])
    if not a["team_id"] or not b["team_id"]:
        return []

    # Ignore anything that is not final or is missing a score
    if not completed:
        return []
    if pd.isna(a["points_for"]) or pd.isna(b["points_for"]):
        return []

    gd = iso_date(game_date)

    return [
        {
            "season_year": SEASON_YEAR,
            "game_date": gd,
            "event_id": eid,
            "commence_time": commence_time,
            "team_id": a["team_id"],
            "team_name": a["team_name"],
            "opponent_id": b["team_id"],
            "opponent_name": b["team_name"],
            "home_away": a["home_away"],
            "neutral_site": neutral_site,
            "points_for": a["points_for"],
            "points_against": b["points_for"],
            "game_completed": completed,
            "status_desc": status_desc,
        },
        {
            "season_year": SEASON_YEAR,
            "game_date": gd,
            "event_id": eid,
            "commence_time": commence_time,
            "team_id": b["team_id"],
            "team_name": b["team_name"],
            "opponent_id": a["team_id"],
            "opponent_name": a["team_name"],
            "home_away": b["home_away"],
            "neutral_site": neutral_site,
            "points_for": b["points_for"],
            "points_against": a["points_for"],
            "game_completed": completed,
            "status_desc": status_desc,
        },
    ]


def ingest_games_for_date(game_date: str, sleep_s: float = 0.0) -> pd.DataFrame:
    """
    Pull ESPN scoreboard data for one date and append any new completed games
    into the season master table.

    Deduping is done by (season_year, event_id, team_id).
    """
    game_date = iso_date(game_date)
    master = load_master()

    if master.empty:
        existing = set()
    else:
        existing = set(
            zip(
                master["season_year"].astype(int),
                master["event_id"].astype(str),
                master["team_id"].astype(str),
            )
        )

    events = fetch_scoreboard_all(game_date).get("events", [])
    new_rows: List[dict] = []

    for ev in events:
        rows = event_to_team_rows(ev, game_date)

        for r in rows:
            key = (int(r["season_year"]), str(r["event_id"]), str(r["team_id"]))
            if key in existing:
                continue
            new_rows.append(r)

        if sleep_s:
            time.sleep(sleep_s)

    if new_rows:
        add_df = pd.DataFrame(new_rows)
        master = pd.concat([master, add_df], ignore_index=True) if not master.empty else add_df
        master = standardize_master(master)
        save_master(master)

    log_line(
        f"Master ingest {game_date}: +{len(new_rows)} rows | "
        f"master_rows={len(master)} | path={MASTER_PATH}"
    )

    return master


log_line(f"CELL 2 loaded — master ingest ready. MASTER_PATH={MASTER_PATH}")

[2026-03-08 18:36:14] CELL 2 loaded — master ingest ready. MASTER_PATH=C:\Users\djbar\popalgo\cache_cbb\master\season_2026\master_team_games.parquet


In [6]:
# Popalgo — Cell 3 (Possessions auditing)

# This cell standardizes possession counts for team-game rows
# and tags each row with the source used to determine possessions.
#
# Possible sources:
#   given     -> possessions already present and valid
#   boxscore  -> computed from boxscore stats
#   proxy     -> league median pace used when possessions unavailable
#   unknown   -> still unresolved
#
# The helpers here are also used to keep backtests safe by filtering
# data strictly by game_date relative to an as-of cutoff.

from __future__ import annotations

from typing import Any, Optional
import numpy as np
import pandas as pd


# If league pace cannot be computed from real possession rows,
# fall back to a constant so the pipeline can still proceed.
FALLBACK_PROXY_PACE = 70.0

# Sanity bounds to prevent unrealistic possession values
MIN_REASONABLE_POSS = 40.0
MAX_REASONABLE_POSS = 95.0


def filter_master_by_game_date(master: pd.DataFrame, as_of_date: str) -> pd.DataFrame:
    """
    Strict backtest filter: keep only rows where game_date <= as_of_date.
    """
    if master is None or master.empty:
        return master

    ao = pd.to_datetime(as_of_date, errors="coerce")
    if pd.isna(ao):
        return master.copy()

    ao = ao.normalize()

    out = master.copy()
    gd = pd.to_datetime(out.get("game_date", pd.NaT), errors="coerce").dt.normalize()

    return out[gd <= ao].copy()


def master_asof_strict(master: pd.DataFrame, as_of_date: str) -> pd.DataFrame:
    """
    Optional as-of filter using snapshot columns if they exist.
    If no snapshot columns exist, returns the input unchanged.
    """
    if master is None or master.empty:
        return master

    ao = pd.to_datetime(as_of_date, errors="coerce")
    if pd.isna(ao):
        return master.copy()

    ao = ao.normalize()

    candidates = [
        "data_asof_date",
        "asof_date",
        "as_of_date",
        "snapshot_date",
        "snapshot_ts",
        "snapshot_ts_utc",
    ]

    for c in candidates:
        if c in master.columns:
            try:
                m = pd.to_datetime(master[c], errors="coerce")
                return master[m.dt.normalize() <= ao].copy()
            except Exception:
                pass

    return master.copy()


def _finite_poss(s: pd.Series) -> pd.Series:
    s2 = pd.to_numeric(s, errors="coerce")
    return s2.notna() & np.isfinite(s2) & (s2 > 0)


def _reasonable_poss(s: pd.Series) -> pd.Series:
    s2 = pd.to_numeric(s, errors="coerce")
    return _finite_poss(s2) & (s2 >= MIN_REASONABLE_POSS) & (s2 <= MAX_REASONABLE_POSS)


def league_median_pace(df: pd.DataFrame) -> float:
    """
    Estimate league median pace from rows with valid possessions.

    pace_game = 0.5 * (possessions + opp_possessions)
    """
    if df is None or df.empty:
        return float(FALLBACK_PROXY_PACE)

    poss = pd.to_numeric(df.get("possessions", np.nan), errors="coerce")
    opp = pd.to_numeric(df.get("opp_possessions", np.nan), errors="coerce")

    ok = _reasonable_poss(poss) & _reasonable_poss(opp)

    if not bool(ok.any()):
        return float(FALLBACK_PROXY_PACE)

    pace = 0.5 * (poss[ok] + opp[ok])
    pace = pd.to_numeric(pace, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

    if len(pace) == 0:
        return float(FALLBACK_PROXY_PACE)

    med = float(pace.median())

    if not np.isfinite(med):
        return float(FALLBACK_PROXY_PACE)

    return float(np.clip(med, MIN_REASONABLE_POSS, MAX_REASONABLE_POSS))


def ensure_possessions_with_source(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures numeric possessions and tags the source used.

    Priority order:
        1. given
        2. boxscore
        3. proxy
        4. unknown
    """
    out = df.copy()

    out["poss_source"] = "unknown"

    out["possessions"] = pd.to_numeric(out.get("possessions", np.nan), errors="coerce")
    out["opp_possessions"] = pd.to_numeric(out.get("opp_possessions", np.nan), errors="coerce")

    out["possessions_used"] = out["possessions"].copy()
    out["opp_possessions_used"] = out["opp_possessions"].copy()

    # Some pipelines default to 70/70 for missing possessions.
    # Treat those as missing so they don't contaminate pace estimates.
    placeholder_7070 = (
        out["possessions_used"].notna()
        & out["opp_possessions_used"].notna()
        & (out["possessions_used"].round(6) == float(FALLBACK_PROXY_PACE))
        & (out["opp_possessions_used"].round(6) == float(FALLBACK_PROXY_PACE))
    )

    out.loc[placeholder_7070, ["possessions_used", "opp_possessions_used"]] = np.nan
    out.loc[placeholder_7070, "poss_source"] = "unknown"

    # Tier 1: real possessions already present
    given_mask = _reasonable_poss(out["possessions_used"]) & _reasonable_poss(
        out["opp_possessions_used"]
    )

    out.loc[given_mask, "poss_source"] = "given"

    # Tier 2: derive from boxscore stats
    required = ["FGA", "OReb", "TO", "FTA", "opp_FGA", "opp_OReb", "opp_TO", "opp_FTA"]

    if all(col in out.columns for col in required):

        for c in required:
            out[c] = pd.to_numeric(out[c], errors="coerce")

        needs_box = out["poss_source"].eq("unknown")
        has_signal = out[required].notna().all(axis=1)
        nonneg = (out[required] >= 0).all(axis=1)

        box_mask = needs_box & has_signal & nonneg

        poss_calc = out["FGA"] - out["OReb"] + out["TO"] + 0.475 * out["FTA"]

        opp_calc = (
            out["opp_FGA"] - out["opp_Oreb"] + out["opp_TO"] + 0.475 * out["opp_FTA"]
            if "opp_Oreb" in out.columns
            else out["opp_FGA"] - out["opp_OReb"] + out["opp_TO"] + 0.475 * out["opp_FTA"]
        )

        out.loc[box_mask, "possessions_used"] = out.loc[box_mask, "possessions_used"].fillna(
            poss_calc[box_mask]
        )

        out.loc[box_mask, "opp_possessions_used"] = out.loc[
            box_mask, "opp_possessions_used"
        ].fillna(opp_calc[box_mask])

        now_ok = _reasonable_poss(out["possessions_used"]) & _reasonable_poss(
            out["opp_possessions_used"]
        )

        out.loc[box_mask & now_ok, "poss_source"] = "boxscore"

    # Tier 3: proxy pace (record only, do not overwrite possessions)

    out["points_for"] = pd.to_numeric(out.get("points_for", np.nan), errors="coerce")
    out["points_against"] = pd.to_numeric(out.get("points_against", np.nan), errors="coerce")

    out["proxy_pace"] = np.nan

    needs_proxy = out["poss_source"].eq("unknown")
    has_points = out["points_for"].notna() & out["points_against"].notna()

    proxy_ok = needs_proxy & has_points

    if bool(proxy_ok.any()):

        proxy_pace = league_median_pace(out)

        out.loc[proxy_ok, "proxy_pace"] = float(proxy_pace)
        out.loc[proxy_ok, "poss_source"] = "proxy"

    # Final sanity check
    bad = ~(
        _reasonable_poss(out["possessions_used"])
        & _reasonable_poss(out["opp_possessions_used"])
    )

    out.loc[bad, ["possessions_used", "opp_possessions_used"]] = np.nan
    out.loc[bad, "poss_source"] = "unknown"

    out["possessions"] = out["possessions_used"]
    out["opp_possessions"] = out["opp_possessions_used"]

    return out


def poss_source_summary(
    slate_games: pd.DataFrame,
    SLATE_DATE: str,
    AS_OF_DATE: str,
    write: bool = True,
) -> pd.DataFrame:
    """
    Produce a quick summary of possession sources for a slate.
    Optionally saves a CSV to eva_cache for auditing.
    """
    df = ensure_possessions_with_source(slate_games.copy())

    df["poss_source"] = df["poss_source"].fillna("unknown")

    summary = (
        df.groupby("poss_source", dropna=False)
        .size()
        .reset_index(name="team_game_rows")
        .sort_values("team_game_rows", ascending=False)
        .reset_index(drop=True)
    )

    summary["slate_date"] = SLATE_DATE
    summary["data_asof_date"] = AS_OF_DATE

    total = int(summary["team_game_rows"].sum())
    summary["pct"] = summary["team_game_rows"] / total if total else 0.0

    proxy_pace = league_median_pace(df)
    summary.attrs["proxy_pace_used"] = proxy_pace

    if write:

        out_path = EVA_CACHE_DIR / f"poss_source_summary_{SLATE_DATE}_asof_{AS_OF_DATE}.csv"

        out_path.parent.mkdir(parents=True, exist_ok=True)

        summary.to_csv(out_path, index=False)

        log_line(
            f"Saved possessions source summary: {out_path.name} | proxy_pace={proxy_pace:.2f}"
        )

    return summary


log_line("CELL 3 loaded — possessions auditing ready.")

[2026-03-08 18:37:27] CELL 3 loaded — possessions auditing ready.


In [7]:
# Popalgo — Cell 4 (run_slate)

# This cell builds the daily slate in a strict backtest-safe way.
#
# Main jobs:
# - make sure the season master is populated through AS_OF_DATE
# - pull the slate from ESPN for SLATE_DATE
# - build team snapshots using only history available through AS_OF_DATE
# - keep everything leakage-safe
# - write the daily outputs to the season cache

from __future__ import annotations

import json
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# Default starting point for building the season master if it needs to be backfilled
MASTER_START_DATE_DEFAULT = "2025-11-01"


def ensure_master_range(master_start_date: str, as_of_date: str, sleep_s: float = 0.05) -> pd.DataFrame:
    """
    Make sure the master table includes completed games from the requested
    start date through the as-of date.
    """
    master_start_date = iso_date(master_start_date)
    as_of_date = iso_date(as_of_date)

    for d in date_range_str(master_start_date, as_of_date):
        ingest_games_for_date(d, sleep_s=sleep_s)

    return load_master()


def slate_events_from_scoreboard(slate_date: str) -> pd.DataFrame:
    """
    Pull the slate directly from the ESPN scoreboard.

    This is the authoritative game list for the date, including
    event ids, team ids, and basic matchup metadata.
    """
    sb = fetch_scoreboard_all(slate_date)
    events = sb.get("events", []) or []

    rows = []

    for ev in events:
        event_id = str(ev.get("id") or "")
        comps = ev.get("competitions") or []

        if not comps:
            continue

        comp = comps[0]
        competitors = comp.get("competitors") or []

        home = next((c for c in competitors if (c.get("homeAway") or "").lower() == "home"), None)
        away = next((c for c in competitors if (c.get("homeAway") or "").lower() == "away"), None)

        ht = (home or {}).get("team") or {}
        at = (away or {}).get("team") or {}

        rows.append(
            {
                "slate_date": iso_date(slate_date),
                "event_id": event_id,
                "commence_time": comp.get("date") or ev.get("date"),
                "neutral_site": bool(comp.get("neutralSite", False)),
                "home_id": str(ht.get("id") or ""),
                "away_id": str(at.get("id") or ""),
                "home_team": ht.get("displayName") or ht.get("shortDisplayName"),
                "away_team": at.get("displayName") or at.get("shortDisplayName"),
            }
        )

    df = pd.DataFrame(rows)

    # Normalize empty strings so bad parses are easier to catch
    for c in ["event_id", "home_id", "away_id", "home_team", "away_team"]:
        if c in df.columns:
            df[c] = df[c].replace("", np.nan)

    # Fail fast if scoreboard parsing breaks
    bad = df["home_team"].isna() | df["away_team"].isna()
    if bad.any():
        display(df.loc[bad, ["event_id", "home_id", "away_id", "home_team", "away_team", "commence_time"]].head(10))
        raise RuntimeError(f"ESPN scoreboard parse failed — missing team names for {bad.mean():.0%} of events.")

    return df


def _finite(x) -> bool:
    try:
        return np.isfinite(float(x))
    except Exception:
        return False


def build_team_snapshot(df_team: pd.DataFrame, recent_start: pd.Timestamp, min_games: int) -> Optional[dict]:
    """
    Build one team snapshot using only games available through the as-of cutoff.
    """
    df_team = ensure_possessions_with_source(df_team.copy())

    # With the current master table, possessions are not stored directly
    # for every row, so we cannot drop unknown possession rows outright.
    # Instead, we rely on points-based pace proxies when needed.

    for c in ["possessions", "opp_possessions", "points_for", "points_against"]:
        df_team[c] = pd.to_numeric(df_team.get(c, np.nan), errors="coerce")

    ...
    # Since the current master does not carry full boxscore possessions,
    # use total points as a stable pace proxy when possessions are missing.
    # This keeps pace estimates usable without flattening everything to one value.
    if ("possessions" not in df_team.columns) or df_team["possessions"].isna().all():
        poss_proxy = 0.5 * (df_team["points_for"] + df_team["points_against"])
        poss_proxy = poss_proxy.clip(lower=40.0, upper=95.0)

        df_team["possessions"] = poss_proxy
        df_team["opp_possessions"] = poss_proxy

    df_team = df_team[
        (df_team["possessions"] > 0)
        & (df_team["opp_possessions"] > 0)
        & df_team["points_for"].notna()
        & df_team["points_against"].notna()
    ].copy()

    if len(df_team) < min_games:
        return None

    df_team["pace"] = 0.5 * (df_team["possessions"] + df_team["opp_possessions"])
    df_team["off_eff"] = 100 * df_team["points_for"] / df_team["possessions"]
    df_team["def_eff"] = 100 * df_team["points_against"] / df_team["opp_possessions"]

    season = df_team.mean(numeric_only=True)

    recent = df_team[df_team["game_date_dt"] >= recent_start]
    window = recent.mean(numeric_only=True) if not recent.empty else None

    if not all(_finite(season[x]) for x in ["pace", "off_eff", "def_eff"]):
        return None

    return {
        "season_games": int(len(df_team)),
        "season_pace": float(season["pace"]),
        "season_off_eff": float(season["off_eff"]),
        "season_def_eff": float(season["def_eff"]),
        "window_games": int(len(recent)),
        "window_pace": float(window["pace"]) if window is not None else np.nan,
        "window_off_eff": float(window["off_eff"]) if window is not None else np.nan,
        "window_def_eff": float(window["def_eff"]) if window is not None else np.nan,
    }


def daily_out_dir(SLATE_DATE: str, AS_OF_DATE: str, out_root: Optional[Path] = None) -> Path:
    """
    Build the output folder for one slate run.
    """
    out_root = out_root or (PROJECT_ROOT / "cache_cbb")

    return (
        out_root
        / f"season_{SEASON_YEAR}"
        / f"proj_{SLATE_DATE}"
        / f"dataasof_{AS_OF_DATE}"
        / f"features_{FEATURE_VERSION}"
        / "daily_slate_outputs"
    )


def run_slate(
    SLATE_DATE: str,
    AS_OF_DATE: str,
    master_start_date: str = MASTER_START_DATE_DEFAULT,
    out_root: Optional[Path] = None,
    skip_master_ingest: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Main entry point for building a backtest-safe daily slate.
    """
    SLATE_DATE = iso_date(SLATE_DATE)
    AS_OF_DATE = iso_date(AS_OF_DATE)

    if LEAKAGE_STRICT and normalize_date(AS_OF_DATE) >= normalize_date(SLATE_DATE):
        raise ValueError("Leakage violation: AS_OF_DATE must be < SLATE_DATE")

    log_line(f"RUN_SLATE {SLATE_DATE} | ASOF {AS_OF_DATE}")

    # Step 1: load or build the master table through the as-of date
    if skip_master_ingest:
        master = load_master()
        if master is None or master.empty:
            raise RuntimeError("skip_master_ingest=True but master is missing/empty.")
        log_line("MASTER ingest skipped — using existing master parquet")
    else:
        master = ensure_master_range(master_start_date, AS_OF_DATE)

    master_safe = filter_master_by_game_date(master, AS_OF_DATE)
    master_safe["team_id"] = master_safe["team_id"].astype(str)
    master_safe["game_date_dt"] = pd.to_datetime(master_safe["game_date"], errors="coerce").dt.normalize()

    ...

    # Step 2: pull the slate directly from ESPN
    slate_events = slate_events_from_scoreboard(SLATE_DATE)

    slate_team_ids = set(slate_events["home_id"]).union(set(slate_events["away_id"]))

    # Step 3: keep only historical rows for teams on this slate
    slate_games = master_safe[master_safe["team_id"].isin(slate_team_ids)].copy()

    if slate_games.empty:
        raise RuntimeError("No historical rows found for slate teams.")

    # Step 4: build team snapshots
    recent_start = normalize_date(AS_OF_DATE) - pd.Timedelta(days=LOOKBACK_DAYS)

    # Slightly relax the minimum-game threshold very early in the season
    if normalize_date(AS_OF_DATE) <= normalize_date(master_start_date) + pd.Timedelta(days=45):
        min_games_used = min(MIN_GAMES, 5)
    else:
        min_games_used = MIN_GAMES

    log_line(f"MIN_GAMES_USED={min_games_used}")

    snaps = []

    for tid in tqdm(sorted(slate_team_ids), desc="Building snapshots"):
        df_team = slate_games[slate_games["team_id"] == tid]

        if df_team.empty:
            continue

        snap = build_team_snapshot(df_team, recent_start, min_games_used)

        if snap is None:
            continue

        snaps.append(
            {
                "team_id": tid,
                "proj_date": SLATE_DATE,
                "data_asof_date": AS_OF_DATE,
                **snap,
            }
        )

    snap_df = pd.DataFrame(snaps)

    # Step 5: write outputs for the slate
    OUT_DIR = daily_out_dir(SLATE_DATE, AS_OF_DATE, out_root)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    slate_events.to_parquet(OUT_DIR / "slate_events.parquet", index=False)
    slate_games.drop(columns=["game_date_dt"], errors="ignore").to_parquet(OUT_DIR / "slate_games.parquet", index=False)
    snap_df.to_parquet(OUT_DIR / "slate_snapshot.parquet", index=False)

    meta = {
        "slate_date": SLATE_DATE,
        "as_of_date": AS_OF_DATE,
        "season_year": int(SEASON_YEAR),
        "feature_version": str(FEATURE_VERSION),
        "n_events": int(len(slate_events)),
        "n_snapshots": int(len(snap_df)),
    }

    (OUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))

    log_line(f"run_slate complete — events={len(slate_events)} snapshots={len(snap_df)}")
    log_line(f"Saved: {OUT_DIR}")

    return slate_events, snap_df


log_line("CELL 4 loaded — run_slate ready.")

[2026-03-08 18:41:23] CELL 4 loaded — run_slate ready.


In [8]:
# Popalgo — Cell 5 (EVA daily slate builder)

# This cell builds the game-level projection slate for one day.
# It uses the slate identity from ESPN and the leakage-safe team
# snapshots written by Cell 4, then joins everything together
# into one event-level projection file.
#
# Main jobs:
# - define the slate games from the ESPN scoreboard
# - load the team snapshots produced by Cell 4
# - join home and away snapshot features by team_id
# - compute expected possessions, team scores, total, and spread
# - write the daily projection output to both cache locations
#
# Backtest rule:
# - AS_OF_DATE must always be before SLATE_DATE
#
# Output rule:
# - keep all games even if a projection cannot be built
# - tag each row with projection_status and why_missing_projection

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Force rebuild is useful while iterating so the daily EVA file
# always reflects the latest projection logic.
FORCE_REBUILD_EVA_DAILY = True

# Blend season-level and recent-window team form.
W_SEASON = 0.6
W_WINDOW = 0.4

# Small pace-aware home adjustment in points per possession.
HOME_PPP_BUMP = 0.010   # about +1 point per 100 possessions


def slate_games_from_scoreboard(SLATE_DATE: str) -> pd.DataFrame:
    """
    Build one row per ESPN event for the requested slate date.

    Returned columns include:
      event_id, slate_date, commence_time, home_team, away_team,
      home_id, away_id, neutral_site
    """
    SLATE_DATE = iso_date(SLATE_DATE)
    sb = fetch_scoreboard_all(SLATE_DATE)
    events = sb.get("events", []) or []

    rows = []
    for ev in events:
        eid = str(ev.get("id") or "")
        comps = (ev.get("competitions") or [])
        if not eid or not comps:
            continue

        comp0 = comps[0]
        commence_time = comp0.get("date")
        neutral_site = bool(comp0.get("neutralSite", False))

        competitors = comp0.get("competitors", []) or []
        if len(competitors) != 2:
            continue

        # ESPN usually labels home/away directly.
        # If not, fall back to first and second competitor.
        home = next((c for c in competitors if (c.get("homeAway") or "").lower() == "home"), None)
        away = next((c for c in competitors if (c.get("homeAway") or "").lower() == "away"), None)

        if home is None or away is None:
            home = competitors[0]
            away = competitors[1]

        home_team = (home.get("team") or {})
        away_team = (away.get("team") or {})

        hid = str(home_team.get("id") or "")
        aid = str(away_team.get("id") or "")
        hname = str(home_team.get("displayName") or home_team.get("name") or "")
        aname = str(away_team.get("displayName") or away_team.get("name") or "")

        if not hid or not aid:
            continue

        rows.append({
            "event_id": eid,
            "slate_date": SLATE_DATE,
            "commence_time": commence_time,
            "neutral_site": neutral_site,
            "home_id": hid,
            "away_id": aid,
            "home_team": hname,
            "away_team": aname,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df.drop_duplicates(subset=["event_id"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Keep a parsed UTC timestamp version for downstream sorting and joins.
    df["commence_time_utc"] = pd.to_datetime(df["commence_time"], errors="coerce", utc=True)

    return df


def load_snapshots_from_cell4(SLATE_DATE: str, AS_OF_DATE: str, out_root: Optional[Path] = None) -> pd.DataFrame:
    """
    Load the team snapshot file written by Cell 4 for this slate/as-of pair.
    """
    SLATE_DATE = iso_date(SLATE_DATE)
    AS_OF_DATE = iso_date(AS_OF_DATE)

    OUT_DIR = daily_out_dir(SLATE_DATE, AS_OF_DATE, out_root=out_root)
    p = OUT_DIR / "slate_snapshot.parquet"
    if not p.exists():
        raise FileNotFoundError(f"Missing slate_snapshot.parquet (run Cell 4 first): {p}")

    snap = pd.read_parquet(p).copy()
    if "team_id" not in snap.columns:
        raise KeyError("slate_snapshot missing team_id (unexpected).")

    snap["team_id"] = snap["team_id"].astype(str)

    return snap


def _blend(season_val, last_val, w_last: Optional[float] = None):
    """
    Blend season and recent-window values.

    If one side is missing, use the other side.
    If both are missing, return NaN.
    """
    if w_last is None:
        w_last = W_WINDOW

    try:
        s = float(season_val)
    except Exception:
        s = np.nan
    try:
        l = float(last_val)
    except Exception:
        l = np.nan

    s_ok = np.isfinite(s)
    l_ok = np.isfinite(l)

    if s_ok and l_ok:
        return (1.0 - float(w_last)) * s + float(w_last) * l
    if s_ok:
        return s
    if l_ok:
        return l
    return np.nan


def pace_bucket(pace: float, fast_cut: float, slow_cut: float) -> str:
    """
    Bucket a blended pace value into fast, mid, slow, or unk.
    """
    if not np.isfinite(pace):
        return "unk"
    if pace >= fast_cut:
        return "fast"
    if pace <= slow_cut:
        return "slow"
    return "mid"


def pace_dominance_strength(h_pace: float, a_pace: float) -> float:
    """
    Return a 0 to 1 multiplier for how strongly the slower team
    should influence the expected pace.

    Small pace gaps act more like an average-pace game.
    Large pace gaps allow more slow-team control.
    """
    if not (np.isfinite(h_pace) and np.isfinite(a_pace)):
        return np.nan

    delta = abs(h_pace - a_pace)

    lo, hi = 1.5, 6.0
    if delta <= lo:
        return 0.0
    if delta >= hi:
        return 1.0
    return float((delta - lo) / (hi - lo))


def expected_possessions_v1(
    h_pace: float,
    a_pace: float,
    *,
    fast_cut: float,
    slow_cut: float,
    dict_fast_fast: float = 0.20,
    dict_slow_slow: float = 0.80,
    dict_mixed: float = 0.60,
) -> float:
    """
    Matchup-aware possessions model.

    The slower team has more control than the faster team,
    but the strength of that control depends on the pace matchup.
    """
    if not (np.isfinite(h_pace) and np.isfinite(a_pace)):
        return np.nan

    # If slate-level cutoffs are not usable, fall back to a simple average.
    if not (np.isfinite(fast_cut) and np.isfinite(slow_cut)):
        return float(0.5 * (h_pace + a_pace))

    avgp = 0.5 * (h_pace + a_pace)
    minp = min(h_pace, a_pace)

    hb = pace_bucket(h_pace, fast_cut, slow_cut)
    ab = pace_bucket(a_pace, fast_cut, slow_cut)

    if hb == "fast" and ab == "fast":
        D_base = dict_fast_fast
    elif hb == "slow" and ab == "slow":
        D_base = dict_slow_slow
    else:
        D_base = dict_mixed

    # Reduce dominance when the pace gap is weak,
    # but do not let it fully disappear.
    strength = pace_dominance_strength(h_pace, a_pace)
    if not np.isfinite(strength):
        strength = 0.0

    strength = max(0.25, float(strength))
    D = D_base * strength

    return float(avgp + D * (minp - avgp))


def project_game_row(
    home: pd.Series,
    away: pd.Series,
    *,
    fast_cut: float,
    slow_cut: float,
    dict_fast_fast: float = 0.20,
    dict_slow_slow: float = 0.80,
    dict_mixed: float = 0.60,
) -> Dict[str, Any]:
    """
    Build the core projection outputs for one game row.
    """
    h_pace = _blend(
        home.get("season_pace"),
        home.get(f"last_{LOOKBACK_DAYS}_pace", home.get("window_pace"))
    )
    a_pace = _blend(
        away.get("season_pace"),
        away.get(f"last_{LOOKBACK_DAYS}_pace", away.get("window_pace"))
    )

    exp_poss = expected_possessions_v1(
        h_pace,
        a_pace,
        fast_cut=fast_cut,
        slow_cut=slow_cut,
        dict_fast_fast=dict_fast_fast,
        dict_slow_slow=dict_slow_slow,
        dict_mixed=dict_mixed,
    )

    h_off = _blend(home.get("season_off_eff"), home.get(f"last_{LOOKBACK_DAYS}_off_eff"))
    h_def = _blend(home.get("season_def_eff"), home.get(f"last_{LOOKBACK_DAYS}_def_eff"))
    a_off = _blend(away.get("season_off_eff"), away.get(f"last_{LOOKBACK_DAYS}_off_eff"))
    a_def = _blend(away.get("season_def_eff"), away.get(f"last_{LOOKBACK_DAYS}_def_eff"))

    home_ppp = np.nan
    away_ppp = np.nan
    if np.isfinite(h_off) and np.isfinite(a_def):
        home_ppp = 0.5 * (h_off + a_def) / 100.0
    if np.isfinite(a_off) and np.isfinite(h_def):
        away_ppp = 0.5 * (a_off + h_def) / 100.0

    home_exp = np.nan
    away_exp = np.nan

    if np.isfinite(exp_poss) and np.isfinite(home_ppp):
        home_exp = exp_poss * (home_ppp + HOME_PPP_BUMP)

    if np.isfinite(exp_poss) and np.isfinite(away_ppp):
        away_exp = exp_poss * away_ppp

    exp_total = home_exp + away_exp if np.isfinite(home_exp) and np.isfinite(away_exp) else np.nan
    home_proj_spread = (home_exp - away_exp) * -1 if np.isfinite(home_exp) and np.isfinite(away_exp) else np.nan

    return dict(
        exp_poss=float(exp_poss) if np.isfinite(exp_poss) else np.nan,
        home_exp=float(home_exp) if np.isfinite(home_exp) else np.nan,
        away_exp=float(away_exp) if np.isfinite(away_exp) else np.nan,
        exp_total=float(exp_total) if np.isfinite(exp_total) else np.nan,
        home_proj_spread=float(home_proj_spread) if np.isfinite(home_proj_spread) else np.nan,
    )


def compute_pace_cutoffs(snap: pd.DataFrame, lookback_days: int) -> tuple[float, float]:
    """
    Compute slate-level fast and slow pace cutoffs from blended team pace.

    If pace values are missing or too compressed, return NaN cutoffs
    so the possessions model falls back safely to average pace.
    """
    s = snap.copy()

    s["pace_blend"] = s.apply(
        lambda r: _blend(
            r.get("season_pace"),
            r.get(f"last_{lookback_days}_pace", r.get("window_pace"))
        ),
        axis=1,
    )

    pace_vals = s["pace_blend"].dropna()

    if len(pace_vals) == 0:
        return (np.nan, np.nan)

    if pace_vals.nunique() <= 1:
        return (np.nan, np.nan)

    fast_cut = float(pace_vals.quantile(0.75))
    slow_cut = float(pace_vals.quantile(0.25))

    if not (np.isfinite(fast_cut) and np.isfinite(slow_cut)):
        return (np.nan, np.nan)
    if abs(fast_cut - slow_cut) < 0.25:
        return (np.nan, np.nan)

    return (fast_cut, slow_cut)


def eva_daily_path(slate_date: str) -> Path:
    """
    Build the flat EVA cache path used by downstream cells.
    """
    ds = pd.to_datetime(slate_date).strftime("%Y%m%d")
    return EVA_CACHE_DIR / f"eva_{SEASON_YEAR}_{ds}.parquet"


def build_eva_daily(
    SLATE_DATE: str,
    AS_OF_DATE: str,
    out_root: Optional[Path] = None,
    force_rebuild: bool = False,
) -> pd.DataFrame:
    """
    Build the daily game-level projection slate.

    Inputs come from Cell 4:
      - slate_events.parquet
      - slate_snapshot.parquet

    Outputs written here:
      - OUT_DIR/slate_projections.parquet
      - EVA_CACHE_DIR/eva_{SEASON_YEAR}_{yyyymmdd}.parquet
    """
    SLATE_DATE = iso_date(SLATE_DATE)
    AS_OF_DATE = iso_date(AS_OF_DATE)

    if LEAKAGE_STRICT and normalize_date(AS_OF_DATE) >= normalize_date(SLATE_DATE):
        raise ValueError("Leakage rule (backtest): AS_OF_DATE must be < SLATE_DATE")

    out_eva = eva_daily_path(SLATE_DATE)
    OUT_DIR = daily_out_dir(SLATE_DATE, AS_OF_DATE, out_root=out_root)
    out_struct = OUT_DIR / "slate_projections.parquet"

    if out_eva.exists() and out_struct.exists() and (not force_rebuild) and (not FORCE_REBUILD_EVA_DAILY):
        log_line(f"⏭️ Cell 5 cache hit: {out_eva.name}")
        return pd.read_parquet(out_eva)

    # Load the authoritative game list from Cell 4.
    events_path = OUT_DIR / "slate_events.parquet"
    if not events_path.exists():
        raise FileNotFoundError(
            f"Missing slate_events.parquet. Run Cell 4 first.\nExpected: {events_path}"
        )

    games = pd.read_parquet(events_path)

    for c in ["event_id", "home_id", "away_id"]:
        if c in games.columns:
            games[c] = games[c].astype(str)

    # Fail fast if slate identity broke upstream.
    bad_names = games["home_team"].isna() | games["away_team"].isna()
    if bad_names.any():
        display(games.loc[bad_names, ["event_id","home_id","away_id","home_team","away_team","commence_time"]].head(10))
        raise RuntimeError(f"Slate events missing team names for {bad_names.mean():.0%} of games.")

    # Load leakage-safe team snapshots from Cell 4.
    snap_path = OUT_DIR / "slate_snapshot.parquet"
    if not snap_path.exists():
        raise FileNotFoundError(
            f"Missing slate_snapshot.parquet. Run Cell 4 first.\nExpected: {snap_path}"
        )

    snap = pd.read_parquet(snap_path)

    if snap.empty:
        log_line("⚠️ Snapshots are empty — projections will be missing for all games.")

    if "team_id" not in snap.columns:
        raise KeyError("Snapshot file missing 'team_id'. Cell 4 must write team_id in slate_snapshot.parquet.")
    snap["team_id"] = snap["team_id"].astype(str)

    # Compute pace cutoffs using only this slate snapshot set.
    FAST_CUT_LOCAL, SLOW_CUT_LOCAL = compute_pace_cutoffs(snap, LOOKBACK_DAYS)

    log_line(
        f"📏 Pace cutoffs — slow ≤ {SLOW_CUT_LOCAL:.2f} | fast ≥ {FAST_CUT_LOCAL:.2f}"
        if np.isfinite(FAST_CUT_LOCAL)
        else "📏 Pace cutoffs — unavailable (fallback mode)"
    )

    # These base snapshot columns are required by the projection formula.
    need_base = ["season_pace", "season_off_eff", "season_def_eff"]
    missing_cols = [c for c in need_base if c not in snap.columns]
    if missing_cols:
        raise KeyError(f"Snapshot columns missing: {missing_cols}. Ensure Cell 4 writes season_pace/off/def.")

    # Prefix snapshots so the merged event table carries both team sides cleanly.
    home_snap = snap.add_prefix("home_").rename(columns={"home_team_id": "home_id"})
    away_snap = snap.add_prefix("away_").rename(columns={"away_team_id": "away_id"})

    # team_id becomes home_team_id / away_team_id after add_prefix,
    # so rename those to the join keys used by the game table.
    if "home_team_id" in home_snap.columns:
        home_snap = home_snap.rename(columns={"home_team_id": "home_id"})
    if "away_team_id" in away_snap.columns:
        away_snap = away_snap.rename(columns={"away_team_id": "away_id"})

    merged = games.merge(home_snap, on="home_id", how="left").merge(away_snap, on="away_id", how="left")

    # Identify which side is missing enough snapshot data to project.
    need_home = ["home_season_pace", "home_season_off_eff", "home_season_def_eff"]
    need_away = ["away_season_pace", "away_season_off_eff", "away_season_def_eff"]

    for c in need_home + need_away:
        if c not in merged.columns:
            merged[c] = np.nan

    home_ok = merged[need_home].notna().all(axis=1)
    away_ok = merged[need_away].notna().all(axis=1)

    why = np.where(~home_ok & ~away_ok, "missing_home_snapshot|missing_away_snapshot",
          np.where(~home_ok, "missing_home_snapshot",
          np.where(~away_ok, "missing_away_snapshot", "")))

    proj_rows = []
    for i, r in merged.iterrows():
        if why[i] != "":
            proj_rows.append({
                "home_exp": np.nan,
                "away_exp": np.nan,
                "exp_total": np.nan,
                "home_proj_spread": np.nan,
                "exp_poss": np.nan
            })
            continue

        def _finite(x):
            try:
                return np.isfinite(float(x))
            except Exception:
                return False

        def _val(series, key):
            return series.get(key, np.nan)

        # Rebuild side-specific inputs in the shape expected by project_game_row.
        # If explicit last_{LOOKBACK_DAYS}_ fields are missing, fall back to window_*.
        home_in = pd.Series({
            "season_pace": _val(r, "home_season_pace"),
            "season_off_eff": _val(r, "home_season_off_eff"),
            "season_def_eff": _val(r, "home_season_def_eff"),
            f"last_{LOOKBACK_DAYS}_pace": _val(r, "home_window_pace") if f"home_last_{LOOKBACK_DAYS}_pace" not in merged.columns else _val(r, f"home_last_{LOOKBACK_DAYS}_pace"),
            f"last_{LOOKBACK_DAYS}_off_eff": _val(r, "home_window_off_eff") if f"home_last_{LOOKBACK_DAYS}_off_eff" not in merged.columns else _val(r, f"home_last_{LOOKBACK_DAYS}_off_eff"),
            f"last_{LOOKBACK_DAYS}_def_eff": _val(r, "home_window_def_eff") if f"home_last_{LOOKBACK_DAYS}_def_eff" not in merged.columns else _val(r, f"home_last_{LOOKBACK_DAYS}_def_eff"),
        })

        away_in = pd.Series({
            "season_pace": _val(r, "away_season_pace"),
            "season_off_eff": _val(r, "away_season_off_eff"),
            "season_def_eff": _val(r, "away_season_def_eff"),
            f"last_{LOOKBACK_DAYS}_pace": _val(r, "away_window_pace") if f"away_last_{LOOKBACK_DAYS}_pace" not in merged.columns else _val(r, f"away_last_{LOOKBACK_DAYS}_pace"),
            f"last_{LOOKBACK_DAYS}_off_eff": _val(r, "away_window_off_eff") if f"away_last_{LOOKBACK_DAYS}_off_eff" not in merged.columns else _val(r, f"away_last_{LOOKBACK_DAYS}_off_eff"),
            f"last_{LOOKBACK_DAYS}_def_eff": _val(r, "away_window_def_eff") if f"away_last_{LOOKBACK_DAYS}_def_eff" not in merged.columns else _val(r, f"away_last_{LOOKBACK_DAYS}_def_eff"),
        })

        proj = project_game_row(
            home=home_in,
            away=away_in,
            fast_cut=FAST_CUT_LOCAL,
            slow_cut=SLOW_CUT_LOCAL,
            dict_fast_fast=0.20,
            dict_slow_slow=0.80,
            dict_mixed=0.60,
        )

        # Keep audit fields so matchup pace behavior is easy to inspect later.
        h_pace_b = _blend(home_in.get("season_pace"), home_in.get(f"last_{LOOKBACK_DAYS}_pace"))
        a_pace_b = _blend(away_in.get("season_pace"), away_in.get(f"last_{LOOKBACK_DAYS}_pace"))

        hb = pace_bucket(h_pace_b, FAST_CUT_LOCAL, SLOW_CUT_LOCAL)
        ab = pace_bucket(a_pace_b, FAST_CUT_LOCAL, SLOW_CUT_LOCAL)

        proj_rows.append({
            "home_exp": proj.get("home_exp", np.nan),
            "away_exp": proj.get("away_exp", np.nan),
            "exp_total": proj.get("exp_total", np.nan),
            "home_proj_spread": proj.get("home_proj_spread", np.nan),
            "exp_poss": proj.get("exp_poss", np.nan),
            "home_pace_blend": h_pace_b,
            "away_pace_blend": a_pace_b,
            "pace_delta": abs(h_pace_b - a_pace_b) if (np.isfinite(h_pace_b) and np.isfinite(a_pace_b)) else np.nan,
            "pace_strength": pace_dominance_strength(h_pace_b, a_pace_b) if (np.isfinite(h_pace_b) and np.isfinite(a_pace_b)) else np.nan,
            "home_pace_bucket": hb,
            "away_pace_bucket": ab,
            "fast_fast": (hb == "fast" and ab == "fast"),
            "slow_slow": (hb == "slow" and ab == "slow"),
        })

    eva = pd.DataFrame({
        "slate_date": SLATE_DATE,
        "data_asof_date": AS_OF_DATE,
        "season_year": SEASON_YEAR,
        "feature_version": FEATURE_VERSION,
        "event_id": merged.get("event_id"),
        "commence_time": merged.get("commence_time"),
        "commence_time_utc": merged.get("commence_time_utc"),
        "neutral_site": merged.get("neutral_site", False),
        "home_id": merged.get("home_id"),
        "away_id": merged.get("away_id"),
        "home_team": merged.get("home_team"),
        "away_team": merged.get("away_team"),
    })

    eva = pd.concat([eva.reset_index(drop=True), pd.DataFrame(proj_rows).reset_index(drop=True)], axis=1)

    eva["projection_status"] = np.where(why == "", "ok", "missing_projection")
    eva["why_missing_projection"] = why

    # Keep the downstream contract stable for Cells 6, 7, and 8.
    required = ["slate_date","event_id","commence_time","home_team","away_team","home_exp","away_exp"]
    miss = [c for c in required if c not in eva.columns]
    if miss:
        raise KeyError(f"EVA daily missing required columns: {miss}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    eva.to_parquet(out_struct, index=False)

    out_eva.parent.mkdir(parents=True, exist_ok=True)
    eva.to_parquet(out_eva, index=False)

    n_games = len(eva)
    n_ok = int((eva["projection_status"] == "ok").sum())
    log_line(f"✅ Cell 5 EVA daily built: {SLATE_DATE} asof {AS_OF_DATE} | games={n_games} | projected={n_ok}")
    log_line(f"💾 Saved: {out_struct}")
    log_line(f"💾 Saved: {out_eva}")

    return eva


log_line("CELL 5 loaded — EVA daily builder ready.")

[2026-03-08 18:49:15] CELL 5 loaded — EVA daily builder ready.


In [9]:
# Popalgo — Cell 6 (Odds open/close lines + snapshot cache)

# This cell builds the market line file for one slate date.
# It starts from the EVA daily slate written by Cell 5, maps ESPN
# team names to Odds API team names, then pulls historical DraftKings
# snapshots to capture both open and close numbers.
#
# Main jobs:
# - load the slate identity from the EVA daily cache
# - map ESPN team names to Odds API team names using the official crosswalk
# - fetch historical Odds API snapshots at the open checkpoints and close checkpoint
# - match events deterministically with an order-invariant pair key
# - extract DraftKings totals and home-team spreads
# - fall back to close if no open line exists at 12h, 6h, or 3h
# - write one cached parquet for the slate date
#
# Open rule:
# - check 12 hours, then 6 hours, then 3 hours before tip
# - stop at the first snapshot with a usable line
#
# Close rule:
# - use the snapshot from 5 minutes before tip
#
# Cache rule:
# - cache historical snapshot responses in memory and on disk
# - reuse cached snapshots across notebook runs when possible

from __future__ import annotations

from pathlib import Path
import json
import re
import time
import unicodedata
from typing import Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import requests

# DraftKings is the only book used in this cell.
BOOK_KEY = "draftkings"

# Define the open checkpoints in the order they should be checked.
# The first usable line becomes the official open.
OPEN_CHECKPOINT_HOURS = [12, 6, 3]

# Close is defined as five minutes before tip.
CLOSE_MINUTES = 5

# Odds API historical endpoint base.
ODDS_HIST_BASE = "https://api.the-odds-api.com/v4/historical/sports"

# Small delay between snapshot calls.
SNAPSHOT_SLEEP_S = 0.05

# Store raw historical snapshot responses on disk so repeated notebook
# runs do not keep hitting the API for the same timestamps.
SNAPSHOT_DISK_DIR = ODDS_CACHE_DIR / "_snapshots"
SNAPSHOT_DISK_DIR.mkdir(parents=True, exist_ok=True)


def _norm(s) -> str:
    """
    Normalize strings so crosswalk joins and pair keys are stable.
    """
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _pair_key(a, b) -> str | None:
    """
    Build an order-invariant matchup key from two team names.
    """
    a2, b2 = _norm(a), _norm(b)
    if not a2 or not b2:
        return None
    x, y = sorted([a2, b2])
    return f"{x}__{y}"


def _iso_utc(ts: pd.Timestamp) -> str:
    """
    Convert a timestamp into the UTC ISO format expected by Odds API.
    """
    ts = pd.to_datetime(ts)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def _floor_5min(ts: pd.Timestamp) -> pd.Timestamp:
    """
    Round timestamps down to the nearest 5-minute mark so snapshot
    requests are consistent.
    """
    return pd.to_datetime(ts).floor("5min")


_odds_sess = requests.Session()


def http_get_json_with_headers(url: str, params: dict, timeout: int = 45) -> tuple[dict, dict]:
    """
    Request Odds API JSON with retry logic and response headers.
    """
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = _odds_sess.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json(), dict(r.headers)
        except Exception as e:
            last_err = e
            log_line(f"⚠️ ODDS HTTP error {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep((RETRY_BACKOFF ** (attempt - 1)) * max(SNAPSHOT_SLEEP_S, 0.05))
            else:
                raise last_err


def load_official_crosswalk(path: Path) -> pd.DataFrame:
    """
    Load the official ESPN-to-Odds team name crosswalk.

    Expected columns:
      odds_team_name, espn_teamname, espn_teamid
    """
    assert path.exists(), f"Missing OFFICIAL crosswalk: {path}"
    xw = pd.read_csv(path)

    req = ["odds_team_name", "espn_teamname", "espn_teamid"]
    missing = [c for c in req if c not in xw.columns]
    if missing:
        raise KeyError(f"Crosswalk missing columns: {missing}")

    xw = xw[req].copy()
    xw["espn_teamname_norm"] = xw["espn_teamname"].map(_norm)
    xw["odds_team_name_norm"] = xw["odds_team_name"].map(_norm)

    # Keep the first mapping for each normalized ESPN team name.
    xw = xw.drop_duplicates(subset=["espn_teamname_norm"], keep="first")

    log_line(f"[XWALK] Loaded {len(xw)} mappings from {path.name}")
    return xw


def load_slate_from_eva_cache(slate_date: str) -> pd.DataFrame:
    """
    Load the daily slate identity from the EVA cache written by Cell 5.
    """
    ds = pd.to_datetime(slate_date).strftime("%Y%m%d")
    eva_path = EVA_CACHE_DIR / f"eva_{SEASON_YEAR}_{ds}.parquet"
    if not eva_path.exists():
        raise FileNotFoundError(f"Missing EVA daily cache (run Cell 5 first): {eva_path}")

    df = pd.read_parquet(eva_path)

    needed = ["home_team", "away_team", "commence_time", "event_id", "slate_date"]
    for c in needed:
        if c not in df.columns:
            df[c] = np.nan

    slate = df[["event_id", "slate_date", "commence_time", "home_team", "away_team"]].drop_duplicates().copy()
    slate["slate_date"] = slate["slate_date"].fillna(iso_date(slate_date))

    slate["home_team_norm"] = slate["home_team"].map(_norm)
    slate["away_team_norm"] = slate["away_team"].map(_norm)

    # ESPN commence_time is typically ISO, so parse directly into UTC.
    slate["commence_time_utc"] = pd.to_datetime(slate["commence_time"], errors="coerce", utc=True)

    return slate


# Keep a small in-memory cache during the active notebook session.
_snapshot_cache_mem: dict[str, list[dict]] = {}


def _snapshot_disk_path(iso_ts: str) -> Path:
    """
    Build the disk cache path for one historical snapshot timestamp.
    """
    safe = iso_ts.replace(":", "").replace("-", "").replace("T", "_").replace("Z", "Z")
    return SNAPSHOT_DISK_DIR / f"snapshot_{ODDS_SPORT_KEY}_{safe}.json"


def fetch_snapshot_events(iso_ts: str) -> list[dict]:
    """
    Load one historical snapshot's event list.

    Cache order:
      1) memory
      2) disk
      3) API
    """
    if iso_ts in _snapshot_cache_mem:
        return _snapshot_cache_mem[iso_ts]

    p = _snapshot_disk_path(iso_ts)
    if p.exists():
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            events = data.get("events", [])
            if isinstance(events, list):
                _snapshot_cache_mem[iso_ts] = events
                return events
        except Exception:
            pass

    url = f"{ODDS_HIST_BASE}/{ODDS_SPORT_KEY}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": ODDS_REGIONS,
        "markets": ODDS_MARKETS,
        "dateFormat": ODDS_DATE_FORMAT,
        "oddsFormat": ODDS_ODDS_FORMAT,
        "date": iso_ts,
    }
    data, headers = http_get_json_with_headers(url, params=params, timeout=45)

    events = data.get("data", data) or []
    if not isinstance(events, list):
        events = []

    _snapshot_cache_mem[iso_ts] = events

    # Best-effort disk write so the same timestamp can be reused later.
    try:
        payload = {
            "iso_ts": iso_ts,
            "events": events,
            "rate_limit_remaining": headers.get("x-requests-remaining") or headers.get("X-Requests-Remaining"),
            "rate_limit_used": headers.get("x-requests-used") or headers.get("X-Requests-Used"),
        }
        p.write_text(json.dumps(payload), encoding="utf-8")
    except Exception:
        pass

    rem = headers.get("x-requests-remaining") or headers.get("X-Requests-Remaining")
    used = headers.get("x-requests-used") or headers.get("X-Requests-Used")
    if rem is not None or used is not None:
        log_line(f"[ODDS] snapshot {iso_ts} | remaining={rem} used={used}")

    time.sleep(SNAPSHOT_SLEEP_S)
    return events


def snapshot_find_event(events: list[dict], pair_key: str) -> Optional[dict]:
    """
    Find one Odds API event inside a snapshot using the deterministic pair key.
    """
    for ev in events:
        pk = _pair_key(ev.get("home_team"), ev.get("away_team"))
        if pk and pk == pair_key:
            return ev
    return None


def dk_lines_from_event(ev: dict, home_team_odds: str) -> tuple[float, float]:
    """
    Extract DraftKings total and home-team spread from one Odds API event.

    The spread returned here is always the number attached to the
    ESPN home team after name mapping.
    """
    if not isinstance(ev, dict):
        return np.nan, np.nan

    home_team_odds_norm = _norm(home_team_odds)
    if not home_team_odds_norm:
        return np.nan, np.nan

    book = next((b for b in (ev.get("bookmakers") or []) if b.get("key") == BOOK_KEY), None)
    if not book:
        return np.nan, np.nan

    markets = book.get("markets") or []
    m_tot = next((m for m in markets if m.get("key") == "totals"), None)
    m_sp  = next((m for m in markets if m.get("key") == "spreads"), None)

    total = np.nan
    if m_tot:
        for o in (m_tot.get("outcomes") or []):
            p = pd.to_numeric(o.get("point", np.nan), errors="coerce")
            if pd.notna(p):
                total = float(p)
                break

    home_spread = np.nan
    if m_sp:
        for o in (m_sp.get("outcomes") or []):
            name = _norm(o.get("name"))
            desc = _norm(o.get("description"))
            if name == home_team_odds_norm or desc == home_team_odds_norm:
                p = pd.to_numeric(o.get("point", np.nan), errors="coerce")
                if pd.notna(p):
                    home_spread = float(p)
                break

    # Guard against clearly bad spread values before downstream merges.
    if pd.notna(home_spread) and (abs(home_spread) > 60):
        home_spread = np.nan

    return total, home_spread


def build_odds_open_close_for_slate(
    SLATE_DATE: str,
    force_rebuild: bool = False,
) -> pd.DataFrame:
    """
    Build the open and close line file for one slate date.
    """
    SLATE_DATE = iso_date(SLATE_DATE)

    out_path = ODDS_CACHE_DIR / f"odds_lines_{SLATE_DATE}_{BOOK_KEY}_open_close.parquet"
    if out_path.exists() and not force_rebuild:
        log_line(f"✅ Odds cache hit: {out_path.name}")
        return pd.read_parquet(out_path)

    # Load the slate identity from EVA daily.
    slate = load_slate_from_eva_cache(SLATE_DATE)

    # Load the official crosswalk and map ESPN names to Odds API names.
    xw = load_official_crosswalk(OFFICIAL_CROSSWALK_PATH)

    slate = slate.merge(
        xw[["espn_teamname_norm", "odds_team_name"]],
        left_on="home_team_norm",
        right_on="espn_teamname_norm",
        how="left",
    ).rename(columns={"odds_team_name": "home_team_odds"}).drop(columns=["espn_teamname_norm"])

    slate = slate.merge(
        xw[["espn_teamname_norm", "odds_team_name"]],
        left_on="away_team_norm",
        right_on="espn_teamname_norm",
        how="left",
    ).rename(columns={"odds_team_name": "away_team_odds"}).drop(columns=["espn_teamname_norm"])

    slate["pair_key"] = slate.apply(lambda r: _pair_key(r["home_team_odds"], r["away_team_odds"]), axis=1)

    out_rows = []

    for _, r in slate.iterrows():
        ct = r.get("commence_time_utc")
        home_team_odds = r.get("home_team_odds")
        away_team_odds = r.get("away_team_odds")
        pk = r.get("pair_key")

        open_total = open_spread = np.nan
        close_total = close_spread = np.nan

        open_checkpoint_used = np.nan
        open_snapshot_ts_utc = ""
        close_snapshot_ts_utc = ""

        odds_event_id = None
        odds_commence_time_utc = pd.NaT

        why: List[str] = []

        # Skip cleanly if the team-name mapping failed.
        if pd.isna(home_team_odds) or pd.isna(away_team_odds) or (pk is None) or (isinstance(pk, float) and pd.isna(pk)):
            why.append("missing_crosswalk_mapping_or_pair_key")

            out_rows.append({
                "slate_date": SLATE_DATE,
                "event_id": r.get("event_id"),
                "commence_time": r.get("commence_time"),
                "home_team": r.get("home_team"),
                "away_team": r.get("away_team"),
                "home_team_odds": home_team_odds,
                "away_team_odds": away_team_odds,
                "pair_key": pk,
                "odds_event_id": odds_event_id,
                "odds_commence_time_utc": odds_commence_time_utc,
                "open_total": open_total,
                "open_spread_home": open_spread,
                "open_checkpoint_hours_used": open_checkpoint_used,
                "open_snapshot_ts_utc": open_snapshot_ts_utc,
                "close_total": close_total,
                "close_spread_home": close_spread,
                "close_snapshot_ts_utc": close_snapshot_ts_utc,
                "open_fallback_to_close": False,
                "why_missing_odds": "|".join(why),
            })
            continue

        # Commence time is required to know where to sample the market.
        if pd.isna(ct):
            why.append("missing_commence_time_utc_in_eva_cache")

            out_rows.append({
                "slate_date": SLATE_DATE,
                "event_id": r.get("event_id"),
                "commence_time": r.get("commence_time"),
                "home_team": r.get("home_team"),
                "away_team": r.get("away_team"),
                "home_team_odds": home_team_odds,
                "away_team_odds": away_team_odds,
                "pair_key": pk,
                "odds_event_id": odds_event_id,
                "odds_commence_time_utc": odds_commence_time_utc,
                "open_total": open_total,
                "open_spread_home": open_spread,
                "open_checkpoint_hours_used": open_checkpoint_used,
                "open_snapshot_ts_utc": open_snapshot_ts_utc,
                "close_total": close_total,
                "close_spread_home": close_spread,
                "close_snapshot_ts_utc": close_snapshot_ts_utc,
                "open_fallback_to_close": False,
                "why_missing_odds": "|".join(why),
            })
            continue

        ct = pd.to_datetime(ct, utc=True, errors="coerce")

        # Always try to capture the close first at tip minus five minutes.
        close_ts = _floor_5min(ct - pd.Timedelta(minutes=CLOSE_MINUTES))
        close_iso = _iso_utc(close_ts)
        close_snapshot_ts_utc = close_iso

        events = fetch_snapshot_events(close_iso)
        ev = snapshot_find_event(events, pk)
        if ev is None:
            why.append("close_snapshot_missing_event_for_pair_key")
        else:
            odds_event_id = str(ev.get("id") or odds_event_id)
            odds_commence_time_utc = pd.to_datetime(ev.get("commence_time"), utc=True, errors="coerce")
            t, s = dk_lines_from_event(ev, home_team_odds=home_team_odds)
            if pd.notna(t):
                close_total, close_spread = t, s
            else:
                why.append("close_snapshot_missing_dk_or_market")

        # Open is the earliest usable line found at 12h, then 6h, then 3h.
        open_found = False
        for h in OPEN_CHECKPOINT_HOURS:
            ts = _floor_5min(ct - pd.Timedelta(hours=h))
            iso_ts = _iso_utc(ts)

            events = fetch_snapshot_events(iso_ts)
            ev = snapshot_find_event(events, pk)
            if ev is None:
                continue

            t, s = dk_lines_from_event(ev, home_team_odds=home_team_odds)
            if pd.notna(t):
                open_total, open_spread = t, s
                open_checkpoint_used = float(h)
                open_snapshot_ts_utc = iso_ts
                open_found = True

                odds_event_id = str(ev.get("id") or odds_event_id)
                odds_commence_time_utc = pd.to_datetime(ev.get("commence_time"), utc=True, errors="coerce")
                break

        # If no open was found, carry the close back into the open fields.
        open_fallback_to_close = False
        if not open_found:
            open_fallback_to_close = True
            open_total, open_spread = close_total, close_spread
            open_checkpoint_used = np.nan
            open_snapshot_ts_utc = close_snapshot_ts_utc
            why.append("open_missing_12_6_3h_fallback_to_close")

        out_rows.append({
            "slate_date": SLATE_DATE,
            "event_id": r.get("event_id"),
            "commence_time": r.get("commence_time"),
            "home_team": r.get("home_team"),
            "away_team": r.get("away_team"),
            "home_team_odds": home_team_odds,
            "away_team_odds": away_team_odds,
            "pair_key": pk,
            "odds_event_id": odds_event_id,
            "odds_commence_time_utc": odds_commence_time_utc,
            "open_total": open_total,
            "open_spread_home": open_spread,
            "open_checkpoint_hours_used": open_checkpoint_used,
            "open_snapshot_ts_utc": open_snapshot_ts_utc,
            "close_total": close_total,
            "close_spread_home": close_spread,
            "close_snapshot_ts_utc": close_snapshot_ts_utc,
            "open_fallback_to_close": open_fallback_to_close,
            "why_missing_odds": "|".join([w for w in why if w]),
        })

    odds_df = pd.DataFrame(out_rows)

    # Save one cached parquet for the full slate date.
    out_path.parent.mkdir(parents=True, exist_ok=True)
    odds_df.to_parquet(out_path, index=False)

    log_line(
        f"💾 Saved odds cache: {out_path.name} | rows={len(odds_df)} | "
        f"missing_open_total={int(pd.isna(odds_df['open_total']).sum())} | "
        f"missing_close_total={int(pd.isna(odds_df['close_total']).sum())} | "
        f"open_fallbacks={int(odds_df.get('open_fallback_to_close', False).sum())}"
    )

    try:
        display(odds_df.head(25))
        display(odds_df["why_missing_odds"].value_counts(dropna=False).head(25))
    except Exception:
        pass

    return odds_df


log_line("CELL 6 loaded — ODDS open/close (12/6/3h + close-5m, fallback-to-close) ready.")

[2026-03-08 18:51:49] CELL 6 loaded — ODDS open/close (12/6/3h + close-5m, fallback-to-close) ready.


In [10]:
# Popalgo — Cell 7 (Build + merge final slates)

# This cell builds the final daily slate files used downstream.
# For each slate date, it enforces the strict backtest rule that
# AS_OF_DATE must be the prior day, then makes sure the projection
# slate and odds slate both exist before merging them together.
#
# Main jobs:
# - set AS_OF_DATE = SLATE_DATE - 1 day
# - ensure Cell 4 and Cell 5 outputs exist for the slate
# - ensure Cell 6 odds outputs exist for the slate
# - merge EVA and odds using event_id as the game spine
# - attach actual home, away, and total scores from the master table
# - save one final slate parquet per date
#
# Build policy:
# - auto-build missing prerequisites when needed
# - still save the final slate even if some games are missing odds or projections
#
# Output path:
# - EVA_CACHE_DIR/final_slate_{SEASON_YEAR}_{yyyymmdd}.parquet

from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

# Dates to build through the full backtest window.
DATES = pd.date_range("2026-03-01", "2026-03-01").strftime("%Y-%m-%d").tolist()

# DraftKings is the only book used for the merged final slate.
BOOK = "draftkings"

# Rebuild switches for the different stages of the pipeline.
FORCE_REBUILD_EVA   = True
FORCE_REBUILD_ODDS  = False
FORCE_REBUILD_FINAL = True


def _season_year_endyear(slate_date: str) -> int:
    """
    Convert a slate date to the season label used by the project.

    College basketball seasons are labeled by the ending year.
    """
    d = pd.to_datetime(slate_date)
    return d.year + 1 if d.month >= 11 else d.year


def load_final_slates_range(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Load all final slate files that exist inside a date range.
    """
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    dates = pd.date_range(start=start, end=end, freq="D")

    dfs = []
    for d in dates:
        slate_date = d.strftime("%Y-%m-%d")
        ymd = d.strftime("%Y%m%d")
        season_year = _season_year_endyear(slate_date)
        p = Path(EVA_CACHE_DIR) / f"final_slate_{season_year}_{ymd}.parquet"
        if p.exists():
            dfs.append(pd.read_parquet(p))

    if not dfs:
        raise ValueError("No final slate files found in date range (path/season_year mismatch or not built).")

    return pd.concat(dfs, ignore_index=True)


def _ds(d: str) -> str:
    """
    Convert a date string to YYYYMMDD form for cache file names.
    """
    return pd.to_datetime(d).strftime("%Y%m%d")


def eva_daily_path(slate_date: str) -> Path:
    """
    Return the EVA daily cache path for one slate date.
    """
    return EVA_CACHE_DIR / f"eva_{SEASON_YEAR}_{_ds(slate_date)}.parquet"


def odds_daily_path(slate_date: str, book: str) -> Path:
    """
    Return the odds cache path for one slate date and book.
    """
    return ODDS_CACHE_DIR / f"odds_lines_{slate_date}_{book}_open_close.parquet"


def final_slate_path(slate_date: str) -> Path:
    """
    Return the final merged slate path for one slate date.
    """
    return EVA_CACHE_DIR / f"final_slate_{SEASON_YEAR}_{_ds(slate_date)}.parquet"


def load_eva_daily(slate_date: str) -> pd.DataFrame:
    """
    Load the EVA daily file for one slate date.
    """
    p = eva_daily_path(slate_date)
    if not p.exists():
        raise FileNotFoundError(f"Missing EVA daily cache: {p}")
    return pd.read_parquet(p)


def load_odds_daily(slate_date: str, book: str) -> pd.DataFrame:
    """
    Load the odds daily file for one slate date.
    """
    p = odds_daily_path(slate_date, book)
    if not p.exists():
        raise FileNotFoundError(f"Missing ODDS daily cache: {p}")
    return pd.read_parquet(p)


def _drop_last_200_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove last_200 columns from the final slate if they leaked through
    from earlier development versions.
    """
    drop_cols = [c for c in df.columns if "last_200" in c]
    if drop_cols:
        log_line(f"🧹 Dropping last_200 cols from FINAL: {len(drop_cols)}")
        df = df.drop(columns=drop_cols, errors="ignore")
    return df


def _assert_final_contract(df: pd.DataFrame) -> None:
    """
    Check that the merged final slate still contains the core downstream contract.
    """
    required = [
        "slate_date", "event_id", "commence_time", "home_team", "away_team",
        "home_exp", "away_exp",
        "open_total", "open_spread_home", "close_total", "close_spread_home",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print("⚠️ Skipping missing FINAL slate parquet(s):")
        for m in missing:
            print("   -", m)


def master_actuals_by_event(as_of_date: str) -> pd.DataFrame:
    """
    Build one row per event_id with home, away, and total actual points.

    Only games with game_date <= as_of_date are included so this stays
    strict-backtest safe.
    """
    as_of_date = iso_date(as_of_date)

    master = load_master().copy()
    if master.empty:
        return pd.DataFrame(columns=["event_id", "home_points_actual", "away_points_actual", "total_actual"])

    master["game_date"] = pd.to_datetime(master["game_date"], errors="coerce")
    cutoff = pd.to_datetime(as_of_date)
    master = master[master["game_date"].dt.normalize() <= cutoff.normalize()].copy()

    master = master[master.get("game_completed", False).astype(bool)].copy()
    master["event_id"] = master["event_id"].astype(str)
    master["home_away"] = master["home_away"].astype(str).str.lower()
    master["points_for"] = pd.to_numeric(master["points_for"], errors="coerce")

    # Only true home and away rows are used here.
    master = master[master["home_away"].isin(["home", "away"])].copy()

    home = master[master["home_away"] == "home"][["event_id", "points_for"]].rename(
        columns={"points_for": "home_points_actual"}
    )
    away = master[master["home_away"] == "away"][["event_id", "points_for"]].rename(
        columns={"points_for": "away_points_actual"}
    )

    act = home.merge(away, on="event_id", how="inner")

    # If duplicate event rows exist, keep the last one.
    act = act.drop_duplicates(subset=["event_id"], keep="last").reset_index(drop=True)

    act["total_actual"] = act["home_points_actual"] + act["away_points_actual"]
    return act


def merge_eva_odds(eva: pd.DataFrame, odds: pd.DataFrame) -> pd.DataFrame:
    """
    Merge EVA and odds on event_id.

    event_id is treated as the one-to-one game spine for the final slate.
    """
    for c in ["event_id", "home_team", "away_team"]:
        if c not in eva.columns:
            raise KeyError(f"EVA missing required column: {c}")
        if c not in odds.columns:
            raise KeyError(f"ODDS missing required column: {c}")

    eva = eva.copy()
    odds = odds.copy()

    eva["event_id"] = eva["event_id"].astype(str)
    odds["event_id"] = odds["event_id"].astype(str)

    eva_dups = int(eva.duplicated(subset=["event_id"]).sum())
    odds_dups = int(odds.duplicated(subset=["event_id"]).sum())
    if eva_dups:
        raise AssertionError(f"EVA has duplicate event_id rows: {eva_dups}")
    if odds_dups:
        raise AssertionError(f"ODDS has duplicate event_id rows: {odds_dups}")

    keep_cols = [
        "event_id",
        "open_total", "open_spread_home", "close_total", "close_spread_home",
        "open_checkpoint_hours_used", "open_snapshot_ts_utc",
        "close_snapshot_ts_utc", "open_fallback_to_close",
        "why_missing_odds",
    ]
    keep_cols = [c for c in keep_cols if c in odds.columns]

    merged = eva.merge(
        odds[keep_cols],
        on=["event_id"],
        how="left",
        validate="one_to_one",
    )

    # Team names may vary slightly by feed, so this merge does not hard-fail on names.
    merged = _drop_last_200_cols(merged)

    for c in ["home_exp", "away_exp"]:
        if c not in merged.columns:
            raise KeyError(f"FINAL missing required projection col '{c}' (EVA schema drift).")

    # Make sure all core odds columns exist even if values are missing.
    for c in ["open_total", "open_spread_home", "close_total", "close_spread_home"]:
        if c not in merged.columns:
            merged[c] = np.nan

    _assert_final_contract(merged)

    return merged


def ensure_eva_daily(slate_date: str, as_of_date: str) -> pd.DataFrame:
    """
    Ensure the EVA daily file exists for one slate date.

    Uses:
    - Cell 4 run_slate
    - Cell 5 build_eva_daily
    """
    slate_date = iso_date(slate_date)
    as_of_date = iso_date(as_of_date)

    p = eva_daily_path(slate_date)
    if p.exists() and not FORCE_REBUILD_EVA:
        log_line(f"⏭️ EVA skip: {p.name}")
        return pd.read_parquet(p)

    log_line(f"▶️ EVA build: slate={slate_date} asof={as_of_date}")

    # Cell 4 builds the leakage-safe slate features and snapshots.
    _ = run_slate(slate_date, as_of_date, skip_master_ingest=True)

    # Cell 5 builds the event-level EVA daily file.
    eva_df = build_eva_daily(slate_date, as_of_date, force_rebuild=True)

    eva_disk = pd.read_parquet(p) if p.exists() else eva_df
    for c in ["event_id", "slate_date", "commence_time", "home_team", "away_team", "home_exp", "away_exp"]:
        if c not in eva_disk.columns:
            raise KeyError(f"EVA cache missing required col '{c}' after build: {p.name}")

    return eva_disk


def ensure_odds_daily(slate_date: str, book: str) -> pd.DataFrame:
    """
    Ensure the odds open/close file exists for one slate date.

    Uses:
    - Cell 6 build_odds_open_close_for_slate
    """
    slate_date = iso_date(slate_date)
    p = odds_daily_path(slate_date, book)

    if p.exists() and not FORCE_REBUILD_ODDS:
        log_line(f"⏭️ ODDS skip: {p.name}")
        return pd.read_parquet(p)

    log_line(f"▶️ ODDS build: {slate_date} ({book}) | force_rebuild={FORCE_REBUILD_ODDS}")
    odds_df = build_odds_open_close_for_slate(slate_date, force_rebuild=True)

    return pd.read_parquet(p) if p.exists() else odds_df


def build_final_slate(slate_date: str, as_of_date: str, book: str, save: bool = True) -> pd.DataFrame:
    """
    Build the final merged slate for one date.

    This merges projections and odds, then attaches actual scores
    from the master table using event_id.
    """
    slate_date = iso_date(slate_date)
    as_of_date = iso_date(as_of_date)

    eva = load_eva_daily(slate_date)
    odds = load_odds_daily(slate_date, book)

    final = merge_eva_odds(eva, odds)

    # Actuals are pulled using the next-day cutoff so completed games
    # from the slate date are available once they are final.
    actuals_asof = iso_date(pd.to_datetime(slate_date) + pd.Timedelta(days=1))
    actuals = master_actuals_by_event(as_of_date=actuals_asof)

    if not actuals.empty:
        final["event_id"] = final["event_id"].astype(str)
        actuals["event_id"] = actuals["event_id"].astype(str)

        final = final.merge(actuals, on="event_id", how="left", validate="one_to_one")
    else:
        for c in ["home_points_actual", "away_points_actual", "total_actual"]:
            if c not in final.columns:
                final[c] = np.nan

    miss_open = int(final["open_total"].isna().sum()) if "open_total" in final.columns else -1
    miss_close = int(final["close_total"].isna().sum()) if "close_total" in final.columns else -1
    miss_proj = int((final["home_exp"].isna() | final["away_exp"].isna()).sum())
    miss_actuals = int(final["home_points_actual"].isna().sum() | final["away_points_actual"].isna().sum())

    log_line(
        f"FINAL {slate_date}: rows={len(final)} | miss_proj={miss_proj} | "
        f"miss_open_total={miss_open} | miss_close_total={miss_close} | miss_actuals={miss_actuals}"
    )

    if "why_missing_odds" in final.columns:
        try:
            top = final["why_missing_odds"].value_counts(dropna=False).head(8)
            log_line(f"why_missing_odds top:\n{top.to_string()}")
        except Exception:
            pass

    if "why_missing_projection" in final.columns:
        try:
            top = final["why_missing_projection"].value_counts(dropna=False).head(8)
            log_line(f"why_missing_projection top:\n{top.to_string()}")
        except Exception:
            pass

    if save:
        out = final_slate_path(slate_date)
        final.to_parquet(out, index=False)
        log_line(f"💾 FINAL saved: {out.name}")

    return final


# Build the final slate set under the strict backtest rule:
# AS_OF_DATE is always the day before SLATE_DATE.
finals = {}
errors = []

for d in DATES:
    try:
        slate_date = iso_date(d)
        as_of_date = iso_date(pd.to_datetime(slate_date) - pd.Timedelta(days=1))

        # Step 1: make sure the EVA daily file exists.
        _ = ensure_eva_daily(slate_date, as_of_date)

        # Step 2: make sure the odds daily file exists.
        _ = ensure_odds_daily(slate_date, BOOK)

        # Step 3: load or rebuild the final merged slate.
        out = final_slate_path(slate_date)
        if out.exists() and not FORCE_REBUILD_FINAL:
            log_line(f"⏭️ FINAL skip: {out.name}")
            finals[slate_date] = pd.read_parquet(out)
        else:
            log_line(f"▶️ FINAL build: {slate_date}")
            finals[slate_date] = build_final_slate(slate_date, as_of_date, BOOK, save=True)

        try:
            cols_peek = [c for c in [
                "slate_date", "event_id", "home_team", "away_team",
                "home_exp", "away_exp",
                "open_total", "open_spread_home", "close_total", "close_spread_home",
                "open_checkpoint_hours_used", "open_fallback_to_close",
                "why_missing_odds", "why_missing_projection"
            ] if c in finals[slate_date].columns]
            display(finals[slate_date][cols_peek].head(15))
        except Exception:
            pass

    except Exception as e:
        log_line(f"❌ Failed {d}: {e}")
        errors.append((d, repr(e)))

if errors:
    msg = "Some dates failed in Cell 7:\n" + "\n".join([f"- {d}: {err}" for d, err in errors])
    raise RuntimeError(msg)

log_line(f"✅ CELL 7 complete — final slates ready for: {DATES[0]} → {DATES[-1]}")

[2026-03-08 18:55:25] ▶️ EVA build: slate=2026-03-01 asof=2026-02-28
[2026-03-08 18:55:25] RUN_SLATE 2026-03-01 | ASOF 2026-02-28
[2026-03-08 18:55:26] MASTER ingest skipped — using existing master parquet
[2026-03-08 18:55:28] MIN_GAMES_USED=3


Building snapshots:   0%|          | 0/46 [00:00<?, ?it/s]

[2026-03-08 18:55:30] run_slate complete — events=23 snapshots=46
[2026-03-08 18:55:30] Saved: C:\Users\djbar\popalgo\cache_cbb\season_2026\proj_2026-03-01\dataasof_2026-02-28\features_v1\daily_slate_outputs
[2026-03-08 18:55:30] 📏 Pace cutoffs — slow ≤ 70.41 | fast ≥ 74.89
[2026-03-08 18:55:30] ✅ Cell 5 EVA daily built: 2026-03-01 asof 2026-02-28 | games=23 | projected=23
[2026-03-08 18:55:30] 💾 Saved: C:\Users\djbar\popalgo\cache_cbb\season_2026\proj_2026-03-01\dataasof_2026-02-28\features_v1\daily_slate_outputs\slate_projections.parquet
[2026-03-08 18:55:30] 💾 Saved: C:\Users\djbar\popalgo\eva_cache\eva_2026_20260301.parquet
[2026-03-08 18:55:30] ⏭️ ODDS skip: odds_lines_2026-03-01_draftkings_open_close.parquet
[2026-03-08 18:55:30] ▶️ FINAL build: 2026-03-01
[2026-03-08 18:55:31] FINAL 2026-03-01: rows=23 | miss_proj=0 | miss_open_total=7 | miss_close_total=7 | miss_actuals=0
[2026-03-08 18:55:31] why_missing_odds top:
why_missing_odds
                                              

,slate_date,event_id,home_team,away_team,home_exp,away_exp,open_total,open_spread_home,close_total,close_spread_home,open_checkpoint_hours_used,open_fallback_to_close,why_missing_odds,why_missing_projection
0,2026-03-01,401825552,Ohio State Buckeyes,Purdue Boilermakers,75.114087,77.917913,150.5,6.5,150.5,6.5,NaN,False,,
1,2026-03-01,401825550,Indiana Hoosiers,Michigan State Spartans,72.719158,75.005956,144.5,3.5,144.5,3.5,NaN,False,,
2,2026-03-01,401828454,Davidson Wildcats,La Salle Explorers,72.954833,66.905274,134.5,-10.5,134.5,-10.5,NaN,False,,
3,2026-03-01,401828254,South Florida Bulls,Tulane Green Wave,79.067627,72.485608,155.5,-14.5,155.5,-14.5,NaN,False,,
4,2026-03-01,401828250,UAB Blazers,North Texas Mean Green,71.944990,70.106938,NaN,NaN,NaN,NaN,NaN,False,open_not_found_in_daily_snapshots|close_not_fo...,
5,2026-03-01,401825551,Maryland Terrapins,Rutgers Scarlet Knights,73.927386,73.804922,141.5,-4.5,141.5,-4.5,NaN,False,,
6,2026-03-01,401830316,Indiana State Sycamores,UIC Flames,72.247544,75.297360,143.5,3.5,143.5,3.5,NaN,False,,
7,2026-03-01,401813372,Canisius Golden Griffins,Quinnipiac Bobcats,66.035706,71.008470,NaN,NaN,NaN,NaN,NaN,False,open_not_found_in_daily_snapshots|close_not_fo...,
8,2026-03-01,401830221,Bradley Braves,Murray State Racers,77.133792,76.219237,158.5,-4.5,158.5,-4.5,NaN,False,,
9,2026-03-01,401828253,Temple Owls,Rice Owls,74.993337,72.330690,141.5,-7.5,141.5,-7.5,NaN,False,,


[2026-03-08 18:55:31] ✅ CELL 7 complete — final slates ready for: 2026-03-01 → 2026-03-01
